# Impacto absoluto — VAEP da dupla por 90 com proxy real

Mede a conexão entre pares comparando o VAEP da dupla por 90 minutos quando A joga com B contra o VAEP por 90 de A jogando com o proxy real de B nas partidas em que B ficou fora.

A leitura é direcionada: `Messi → Suárez` compara `Messi + Suárez` com `Messi + proxy de Suárez`. A direção inversa compara `Suárez + Messi` com `Suárez + proxy de Messi`.

O foco aqui é comparabilidade: somente combinações com pelo menos 300 minutos juntos e pelo menos 180 minutos na condição separada de ausência/proxy entram na tabela final.


## 1. Ambiente

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.synergy_vaep_pipeline import PipelineConfig, run_pipeline

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

PROJECT_ROOT

PosixPath('/Users/gcarninatto/Documents/GitHub/Synergy-VAEP')

## 2. Configuração

Por padrão a pipeline reutiliza `data/processed/vaep_actions.parquet`. Para retreinar o VAEP, use `recompute_vaep=True`.

O notebook usa 300 minutos como corte mínimo para minutos da dupla e 180 minutos como corte mínimo para minutos separados na condição de proxy/ausência.


In [2]:
MIN_TOGETHER_MINUTES = 300
MIN_SEPARATE_PROXY_MINUTES = 180

config = PipelineConfig(
    n_bootstrap=0,
    recompute_vaep=False,
    grid_x_bins=12,
    grid_y_bins=8,
    min_role_events=80,
    min_together_minutes=MIN_TOGETHER_MINUTES,
    min_replacement_minutes=MIN_SEPARATE_PROXY_MINUTES,
)

config.events_path, config.matches_path, config.processed_data_path, config.viz_path


(PosixPath('/Users/gcarninatto/Documents/GitHub/Synergy-VAEP/data/raw/combined/all_events_combined.parquet'),
 PosixPath('/Users/gcarninatto/Documents/GitHub/Synergy-VAEP/data/raw/combined/all_matches_combined.parquet'),
 PosixPath('/Users/gcarninatto/Documents/GitHub/Synergy-VAEP/data/processed'),
 PosixPath('/Users/gcarninatto/Documents/GitHub/Synergy-VAEP/viz'))

## 3. Métrica

`impacto_absoluto_p90(A→B) = VAEP90(A+B juntos) − VAEP90(A+proxy_de_B)`

Para cada combinação direcionada `(A→B)` do mesmo time:

- **A+B juntos**: minutos em que os dois estiveram em campo na mesma equipe.
- **A+proxy de B**: minutos em partidas em que B ficou fora, usando o substituto do mesmo slot ou o jogador mais similar por eigenmap espacial.
- **Impacto positivo**: a dupla real gera mais VAEP por 90 do que a versão com proxy.

O proxy pode variar por partida; `main_proxy_player` mostra o proxy dominante em minutos para facilitar a leitura.


## 4. Execução

In [3]:
outputs = run_pipeline(config)

{
    "directed_synergy": outputs["directed_synergy"].shape,
}


{'directed_synergy': (23286, 52)}

## 5. Impacto absoluto por 90 — resultado

Tabela com todas as combinações que passam no corte de minutos, sem ranking limitado e sem métrica relativa por ação.

A coluna principal é `impacto_absoluto_p90`: quanto a dupla real produz a mais ou a menos que a comparação com proxy, em VAEP por 90 minutos.


In [4]:
import pandas as pd

impacto = outputs.get("directed_synergy")
if impacto is None:
    impacto = pd.read_parquet(config.processed_data_path / "directed_synergy_all.parquet")

impacto_absoluto_p90 = impacto.loc[
    impacto["minutes_together"].ge(MIN_TOGETHER_MINUTES)
    & impacto["minutes_proxy"].ge(MIN_SEPARATE_PROXY_MINUTES)
].copy()

impacto_absoluto_p90["impacto_absoluto_p90"] = (
    impacto_absoluto_p90["duo_vaep90_together"] - impacto_absoluto_p90["proxy_duo_vaep90"]
)
impacto_absoluto_p90["dupla_real"] = (
    impacto_absoluto_p90["focal_player"] + " + " + impacto_absoluto_p90["partner_player"]
)
impacto_absoluto_p90["dupla_proxy"] = (
    impacto_absoluto_p90["focal_player"]
    + " + "
    + impacto_absoluto_p90["main_proxy_player"].fillna("proxy indefinido")
)

impacto_absoluto_p90 = impacto_absoluto_p90.sort_values(
    ["team", "focal_player", "impacto_absoluto_p90"],
    ascending=[True, True, False],
).reset_index(drop=True)

impact_cols = [
    "team",
    "focal_player",
    "partner_player",
    "main_proxy_player",
    "dupla_real",
    "dupla_proxy",
    "impacto_absoluto_p90",
    "duo_vaep90_together",
    "proxy_duo_vaep90",
    "minutes_together",
    "minutes_proxy",
    "matches_together",
    "matches_proxy",
    "duo_actions_together",
    "proxy_duo_actions",
    "mean_proxy_similarity",
    "main_proxy_method",
]
impact_cols = [col for col in impact_cols if col in impacto_absoluto_p90.columns]

impacto_absoluto_p90[impact_cols].round(
    {
        "impacto_absoluto_p90": 3,
        "duo_vaep90_together": 3,
        "proxy_duo_vaep90": 3,
        "minutes_together": 1,
        "minutes_proxy": 1,
        "mean_proxy_similarity": 3,
    }
)


,team,focal_player,partner_player,main_proxy_player,dupla_real,dupla_proxy,impacto_absoluto_p90,duo_vaep90_together,proxy_duo_vaep90,minutes_together,minutes_proxy,matches_together,matches_proxy,duo_actions_together,proxy_duo_actions,mean_proxy_similarity,main_proxy_method
0,AC Milan,Alessio Cerci,Ignazio Abate,Mattia De Sciglio,Alessio Cerci + Ignazio Abate,Alessio Cerci + Mattia De Sciglio,0.054,0.120,0.065,487.8,279.2,9,4,1369.0,669.0,0.373,starter_similarity_slot
1,AC Milan,Alessio Cerci,Mattia De Sciglio,Luca Antonelli,Alessio Cerci + Mattia De Sciglio,Alessio Cerci + Luca Antonelli,-0.050,0.037,0.087,370.8,412.8,9,6,967.0,1091.0,0.177,starter_similarity_slot
2,AC Milan,Alessio Romagnoli,Giacomo Bonaventura,José Mauri,Alessio Romagnoli + Giacomo Bonaventura,Alessio Romagnoli + José Mauri,0.321,0.130,-0.191,2519.7,616.1,29,10,6888.0,1660.0,-0.027,starter_similarity_slot
3,AC Milan,Alessio Romagnoli,Philippe Mexès,Alex Rodrigo Dias da Costa,Alessio Romagnoli + Philippe Mexès,Alessio Romagnoli + Alex Rodrigo Dias da Costa,0.271,0.239,-0.032,327.7,1960.1,4,22,700.0,4582.0,0.106,starter_similarity_slot
4,AC Milan,Alessio Romagnoli,Alessio Cerci,Mario Barwuah Balotelli,Alessio Romagnoli + Alessio Cerci,Alessio Romagnoli + Mario Barwuah Balotelli,0.124,0.088,-0.037,721.0,786.5,11,14,1651.0,1717.0,0.194,starter_similarity_slot
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23281,Wolfsburg,Sebastian Jung,André Schürrle,Nicklas Bendtner,Sebastian Jung + André Schürrle,Sebastian Jung + Nicklas Bendtner,0.088,0.154,0.066,342.0,215.6,7,7,884.0,599.0,0.039,starter_similarity_slot
23282,Wolfsburg,Timm Klose,Adelino André Vieira Freitas,Christian Träsch,Timm Klose + Adelino André Vieira Freitas,Timm Klose + Christian Träsch,0.398,0.254,-0.144,431.9,224.5,6,4,1477.0,738.0,0.471,starter_similarity_slot
23283,Wolfsburg,Timm Klose,Bas Dost,Nicklas Bendtner,Timm Klose + Bas Dost,Timm Klose + Nicklas Bendtner,0.371,0.470,0.099,400.7,254.5,7,7,928.0,678.0,0.218,same_substitution_slot
23284,Wolfsburg,Timm Klose,Josuha Guilavogui,Aaron Hunt,Timm Klose + Josuha Guilavogui,Timm Klose + Aaron Hunt,-0.120,-0.027,0.093,423.6,232.7,7,4,1416.0,648.0,0.099,starter_similarity_slot
